# Global Weather Trend Forecasting
### Tech Assessment for Data Scientist / Analyst — PM Accelerator

---

### 📌 About PM Accelerator

> **Action required before you submit:** paste the official PM Accelerator mission
> statement here, copied verbatim from **pmaccelerator.io** or the PM Accelerator
> LinkedIn company page. The assessment explicitly requires the mission statement
> to be displayed in your final report/dashboard, so it must come from the source,
> not be paraphrased.

PM Accelerator (PMA) is a product-management training and mentorship organization
founded by Dr. Nancy Li. Learn more: https://www.pmaccelerator.io/ and on
[LinkedIn](https://www.linkedin.com/company/product-manager-accelerator/).

---

## Project Objective
Analyze the **Global Weather Repository** dataset (40+ daily weather & air-quality
features for cities worldwide) to forecast weather trends and demonstrate data
science skills — covering **both** the Basic and Advanced assessment tracks.

## Notebook Roadmap
| # | Section | Track |
|---|---------|-------|
| 1 | Setup & Data Loading | — |
| 2 | Data Cleaning & Preprocessing | Basic |
| 3 | Exploratory Data Analysis | Basic |
| 4 | Baseline Forecasting Model + Evaluation | Basic |
| 5 | Advanced EDA — Anomaly Detection | Advanced |
| 6 | Multi-Model Forecasting + Ensemble | Advanced |
| 7 | Unique Analyses (Climate / Air Quality / Feature Importance / Spatial / Geographical) | Advanced |
| 8 | Summary, Insights & Deliverable Checklist | — |


## 1. Setup & Data Loading

Core libraries (`pandas`, `numpy`, `scikit-learn`, `statsmodels`, `xgboost`) are
pre-installed on Kaggle. The cell below installs a few optional extras used in the
**advanced** sections (`prophet`, `shap`, `pycountry_convert`). If your Kaggle
notebook has **Internet = Off** (Settings panel, right sidebar), either turn it on
or just skip this cell — every optional library is wrapped in a `try/except`
later, so the notebook degrades gracefully and still runs end-to-end without them.


In [ ]:
# Optional installs (safe to skip if internet is off in Kaggle settings)
import sys, subprocess

def safe_pip_install(pkgs):
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs],
                        check=True, timeout=180)
    except Exception as e:
        print(f"Could not install {pkgs}: {e}")

safe_pip_install(["prophet"])
safe_pip_install(["shap"])
safe_pip_install(["pycountry_convert"])


In [ ]:
import os, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                              r2_score, mean_absolute_percentage_error)
from sklearn.inspection import permutation_importance

import xgboost as xgb
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
RANDOM_STATE = 42

# ---- Optional libraries: detect availability, never hard-fail ----
try:
    from prophet import Prophet
    HAS_PROPHET = True
except ImportError:
    HAS_PROPHET = False
    print("Prophet not available -> Prophet model step will be skipped.")

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("SHAP not available -> SHAP feature-importance step will be skipped.")

try:
    import pycountry_convert as pcc
    HAS_CONTINENT = True
except ImportError:
    HAS_CONTINENT = False
    print("pycountry_convert not available -> continent-level grouping will be skipped.")

print("Setup complete.")


### Load the dataset

This auto-discovers any CSV under `/kaggle/input/` whose filename contains
"weather", so it works regardless of the exact dataset slug you attached. Adjust
`DATA_PATH` manually if you have multiple weather datasets attached.

In [ ]:
DATA_DIR = "/kaggle/input"
candidates = []
for root, _, files in os.walk(DATA_DIR):
    for f in files:
        if f.lower().endswith(".csv") and "weather" in f.lower():
            candidates.append(os.path.join(root, f))

print("Candidate file(s) found:")
for c in candidates:
    print(" -", c)

DATA_PATH = candidates[0] if candidates else \
    "/kaggle/input/global-weather-repository/GlobalWeatherRepository.csv"

df = pd.read_csv(DATA_PATH)
print("\nShape:", df.shape)
df.head()


## 2. Data Cleaning & Preprocessing — *Basic Assessment*

Steps: inspect structure -> handle missing values -> detect & treat outliers ->
normalize key numeric features. We keep an **unscaled** copy (`df_clean`) for
readable EDA and a **scaled** copy (`df_scaled`) for modeling-style preprocessing
demos.

In [ ]:
df.info()


In [ ]:
# Missing value audit
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_report = missing_report[missing_report.missing_count > 0].sort_values(
    "missing_pct", ascending=False)
missing_report


In [ ]:
df_clean = df.copy()

# Parse the key time field used later for time-series work
df_clean["last_updated"] = pd.to_datetime(df_clean["last_updated"], errors="coerce")

# Numeric columns -> median impute (robust to skew/outliers)
num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Categorical columns -> mode impute
cat_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()
for col in cat_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Dropped {before - len(df_clean)} duplicate rows.")
print("Remaining missing values:", df_clean.isnull().sum().sum())
print("Shape after cleaning:", df_clean.shape)


In [ ]:
def detect_outliers_iqr(series, factor=1.5):
    """Return boolean outlier mask + bounds using the IQR rule."""
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - factor * IQR, Q3 + factor * IQR
    return (series < lower) | (series > upper), lower, upper

key_cols = [c for c in ["temperature_celsius", "precip_mm", "humidity",
                         "wind_kph", "pressure_mb"] if c in df_clean.columns]

outlier_summary = {}
for col in key_cols:
    mask, low, up = detect_outliers_iqr(df_clean[col])
    outlier_summary[col] = int(mask.sum())
    # Winsorize (cap) rather than drop -> preserves daily continuity for time series
    df_clean[col] = df_clean[col].clip(lower=low, upper=up)

pd.Series(outlier_summary, name="outliers_capped")


In [ ]:
# Normalize key numeric features (Min-Max scaling) into a separate modeling frame
scaler = MinMaxScaler()
df_scaled = df_clean.copy()
df_scaled[key_cols] = scaler.fit_transform(df_clean[key_cols])
df_scaled[key_cols].describe().T[["min", "max", "mean"]]


## 3. Exploratory Data Analysis — *Basic Assessment*

Trends, correlations, and patterns, plus required visualizations for
**temperature** and **precipitation**.

In [ ]:
df_clean[key_cols].describe().T


In [ ]:
corr_cols = [c for c in df_clean.select_dtypes(include=[np.number]).columns
             if "epoch" not in c.lower()]
plt.figure(figsize=(16, 12))
sns.heatmap(df_clean[corr_cols].corr(), cmap="coolwarm", center=0)
plt.title("Correlation Matrix — All Numeric Features")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(df_clean["temperature_celsius"], kde=True, ax=axes[0], color="tomato")
axes[0].set_title("Temperature Distribution (\u00b0C)")
sns.histplot(df_clean["precip_mm"], kde=True, ax=axes[1], color="steelblue")
axes[1].set_title("Precipitation Distribution (mm)")
plt.tight_layout()
plt.show()


In [ ]:
df_clean["date"] = df_clean["last_updated"].dt.date
daily = df_clean.groupby("date").agg(
    avg_temp=("temperature_celsius", "mean"),
    avg_precip=("precip_mm", "mean")
).reset_index()

fig = px.line(daily, x="date", y="avg_temp",
              title="Global Average Daily Temperature Trend")
fig.show()

fig2 = px.line(daily, x="date", y="avg_precip",
               title="Global Average Daily Precipitation Trend")
fig2.show()


In [ ]:
top_countries = df_clean["country"].value_counts().head(10).index
subset = df_clean[df_clean["country"].isin(top_countries)]

plt.figure(figsize=(14, 6))
sns.boxplot(data=subset, x="country", y="temperature_celsius")
plt.xticks(rotation=45)
plt.title("Temperature Spread by Country (Top 10 by record count)")
plt.tight_layout()
plt.show()


## 4. Baseline Forecasting Model — *Basic Assessment*

We build a global daily-average temperature series from `last_updated`, engineer
calendar + lag features, split **chronologically** (never randomly for time
series), and fit a Linear Regression baseline. Evaluated with MAE, RMSE, MAPE,
and R².

In [ ]:
ts = df_clean.groupby("date")["temperature_celsius"].mean().reset_index()
ts["date"] = pd.to_datetime(ts["date"])
ts = ts.sort_values("date").reset_index(drop=True)

ts["day_of_year"] = ts["date"].dt.dayofyear
ts["month"] = ts["date"].dt.month
ts["day_of_week"] = ts["date"].dt.dayofweek

for lag in [1, 2, 3, 7]:
    ts[f"lag_{lag}"] = ts["temperature_celsius"].shift(lag)
ts["rolling_mean_7"] = ts["temperature_celsius"].shift(1).rolling(7).mean()

ts = ts.dropna().reset_index(drop=True)
print("Time series length after feature engineering:", len(ts))
ts.head()


In [ ]:
FEATURES = ["day_of_year", "month", "day_of_week",
            "lag_1", "lag_2", "lag_3", "lag_7", "rolling_mean_7"]
TARGET = "temperature_celsius"

split_idx = int(len(ts) * 0.8)
train, test = ts.iloc[:split_idx], ts.iloc[split_idx:]

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

print(f"Train size: {len(train)} | Test size: {len(test)}")


In [ ]:
def evaluate(y_true, y_pred, name, results_list):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2 = r2_score(y_true, y_pred)
    print(f"{name:28s} | MAE={mae:6.3f}  RMSE={rmse:6.3f}  MAPE={mape:6.2f}%  R2={r2:6.3f}")
    results_list.append({"model": name, "MAE": mae, "RMSE": rmse, "MAPE": mape, "R2": r2})
    return None

results = []

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
pred_lr = lr_model.predict(X_test)
evaluate(y_test, pred_lr, "Linear Regression (baseline)", results)


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(test["date"], y_test.values, label="Actual", marker="o")
plt.plot(test["date"], pred_lr, label="Predicted (Linear Regression)", marker="x")
plt.legend()
plt.title("Baseline Forecast vs Actual — Global Avg Temperature")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 5. Advanced EDA — Anomaly Detection — *Advanced Assessment*

Two complementary approaches:
1. **Univariate** Z-score flagging on temperature.
2. **Multivariate** Isolation Forest across temperature, humidity, precipitation,
   wind, and pressure jointly — catches anomalies a single-variable rule would
   miss (e.g. a "normal" temperature paired with an impossible humidity/pressure
   combination).

In [ ]:
z_scores = np.abs(stats.zscore(df_clean["temperature_celsius"]))
df_clean["temp_anomaly_zscore"] = z_scores > 3
print("Z-score anomalies (|z|>3):", int(df_clean['temp_anomaly_zscore'].sum()))

iso_features = [c for c in ["temperature_celsius", "humidity", "precip_mm",
                             "wind_kph", "pressure_mb"] if c in df_clean.columns]

iso_forest = IsolationForest(contamination=0.02, random_state=RANDOM_STATE)
df_clean["anomaly_iso"] = iso_forest.fit_predict(df_clean[iso_features])  # -1 = anomaly
print("Isolation Forest anomalies:", int((df_clean['anomaly_iso'] == -1).sum()))


In [ ]:
normal = df_clean[df_clean["anomaly_iso"] == 1]
anomalous = df_clean[df_clean["anomaly_iso"] == -1]

plt.figure(figsize=(14, 6))
plt.scatter(normal["humidity"], normal["temperature_celsius"],
            s=8, alpha=0.3, label="Normal")
plt.scatter(anomalous["humidity"], anomalous["temperature_celsius"],
            s=25, color="red", label="Anomaly")
plt.xlabel("Humidity (%)")
plt.ylabel("Temperature (\u00b0C)")
plt.legend()
plt.title("Isolation Forest Anomaly Detection — Temperature vs Humidity")
plt.show()


## 6. Multi-Model Forecasting + Ensemble — *Advanced Assessment*

We compare a **statistical** model (SARIMAX), a **tree-based boosting** model
(XGBoost), a **tree-based bagging** model (Random Forest), and, if available,
**Prophet**. We then build a simple **averaging ensemble** and check whether it
beats every individual model on RMSE — the classic justification for
ensembling.

In [ ]:
ts_indexed = ts.set_index("date")["temperature_celsius"]
train_s, test_s = ts_indexed.iloc[:split_idx], ts_indexed.iloc[split_idx:]

sarimax_model = SARIMAX(train_s, order=(2, 1, 2), seasonal_order=(1, 1, 1, 7),
                         enforce_stationarity=False, enforce_invertibility=False)
sarimax_fit = sarimax_model.fit(disp=False)
pred_sarimax = sarimax_fit.forecast(steps=len(test_s))
evaluate(test_s.values, pred_sarimax.values, "SARIMAX(2,1,2)(1,1,1,7)", results)


In [ ]:
xgb_model = xgb.XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8,
                              random_state=RANDOM_STATE)
xgb_model.fit(X_train, y_train)
pred_xgb = xgb_model.predict(X_test)
evaluate(y_test, pred_xgb, "XGBoost", results)


In [ ]:
rf_model = RandomForestRegressor(n_estimators=300, max_depth=6,
                                  random_state=RANDOM_STATE)
rf_model.fit(X_train, y_train)
pred_rf = rf_model.predict(X_test)
evaluate(y_test, pred_rf, "Random Forest", results)


In [ ]:
if HAS_PROPHET:
    prophet_df = ts[["date", "temperature_celsius"]].rename(
        columns={"date": "ds", "temperature_celsius": "y"})
    p_train, p_test = prophet_df.iloc[:split_idx], prophet_df.iloc[split_idx:]

    prophet_model = Prophet(daily_seasonality=True, yearly_seasonality=False,
                             weekly_seasonality=True)
    prophet_model.fit(p_train)

    future = prophet_model.make_future_dataframe(periods=len(p_test))
    forecast = prophet_model.predict(future)
    pred_prophet = forecast.set_index("ds").loc[p_test["ds"], "yhat"].values
    evaluate(p_test["y"].values, pred_prophet, "Prophet", results)
else:
    pred_prophet = None
    print("Prophet skipped (not installed).")


In [ ]:
# Simple averaging ensemble of the same-shaped model predictions
ensemble_pred = np.mean([pred_lr, pred_xgb, pred_rf], axis=0)
evaluate(y_test, ensemble_pred, "Ensemble (avg: LR + XGB + RF)", results)

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
results_df


In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=results_df, x="model", y="RMSE", palette="viridis")
plt.title("Model Comparison \u2014 RMSE (lower is better)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 7. Unique Analyses — *Advanced Assessment*

Five required angles: **Climate**, **Environmental Impact**, **Feature
Importance**, **Spatial Analysis**, and **Geographical Patterns**.

### 7.1 Climate Analysis — long-term & seasonal patterns by region

In [ ]:
df_clean["season"] = (df_clean["last_updated"].dt.month % 12 // 3 + 1).map(
    {1: "Winter", 2: "Spring", 3: "Summer", 4: "Fall"})

seasonal = df_clean.groupby(["country", "season"])["temperature_celsius"].mean().reset_index()
top10 = df_clean["country"].value_counts().head(10).index

fig = px.bar(seasonal[seasonal["country"].isin(top10)],
             x="country", y="temperature_celsius", color="season", barmode="group",
             title="Seasonal Average Temperature by Country (Top 10 by record count)")
fig.show()


### 7.2 Environmental Impact — air quality vs weather correlation

In [ ]:
aq_cols = [c for c in df_clean.columns if "air_quality" in c.lower()
           and df_clean[c].dtype != object]
weather_cols = [c for c in ["temperature_celsius", "humidity", "wind_kph",
                             "pressure_mb", "precip_mm"] if c in df_clean.columns]

print("Air-quality columns detected:", aq_cols)

if aq_cols:
    aq_corr = df_clean[aq_cols + weather_cols].corr().loc[aq_cols, weather_cols]
    plt.figure(figsize=(10, max(4, 0.6 * len(aq_cols))))
    sns.heatmap(aq_corr, annot=True, fmt=".2f", cmap="coolwarm")
    plt.title("Air Quality vs Weather Parameters \u2014 Correlation")
    plt.tight_layout()
    plt.show()
else:
    print("No air_quality_* columns found in this dataset version.")


### 7.3 Feature Importance — three complementary techniques

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=FEATURES) \
    .sort_values(ascending=False)
plt.figure(figsize=(10, 5))
importances.plot(kind="barh", color="teal")
plt.title("Random Forest \u2014 Built-in (Gini/MDI) Feature Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
perm_result = permutation_importance(rf_model, X_test, y_test,
                                      n_repeats=10, random_state=RANDOM_STATE)
perm_series = pd.Series(perm_result.importances_mean, index=FEATURES) \
    .sort_values(ascending=False)
plt.figure(figsize=(10, 5))
perm_series.plot(kind="barh", color="darkorange")
plt.title("Permutation Importance (model-agnostic)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
if HAS_SHAP:
    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_test)
    shap.summary_plot(shap_values, X_test, show=True)
else:
    print("SHAP skipped (not installed) \u2014 run '!pip install shap' with internet on to enable.")


### 7.4 Spatial Analysis — geographical visualization of conditions

In [ ]:
latest_per_location = (df_clean.sort_values("last_updated")
                        .groupby("location_name").tail(1))

fig = px.scatter_geo(latest_per_location, lat="latitude", lon="longitude",
                      color="temperature_celsius", hover_name="location_name",
                      color_continuous_scale="thermal",
                      title="Latest Recorded Temperature by Location (Global)")
fig.show()


### 7.5 Geographical Patterns — country & continent comparison

In [ ]:
if HAS_CONTINENT:
    def country_to_continent(name):
        try:
            alpha2 = pcc.country_name_to_country_alpha2(name, cn_name_format="default")
            cont_code = pcc.country_alpha2_to_continent_code(alpha2)
            return pcc.convert_continent_code_to_continent_name(cont_code)
        except Exception:
            return "Unknown"

    df_clean["continent"] = df_clean["country"].apply(country_to_continent)
    continent_summary = df_clean.groupby("continent")[
        ["temperature_celsius", "humidity", "precip_mm"]].mean().round(2)
    display(continent_summary)

    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df_clean[df_clean["continent"] != "Unknown"],
                x="continent", y="temperature_celsius")
    plt.title("Temperature Distribution by Continent")
    plt.tight_layout()
    plt.show()
else:
    print("pycountry_convert not installed \u2014 showing country-level only.")

country_avg = df_clean.groupby("country").agg(
    avg_temp=("temperature_celsius", "mean"),
    avg_humidity=("humidity", "mean"),
    avg_precip=("precip_mm", "mean")
).reset_index()

fig = px.choropleth(country_avg, locations="country", locationmode="country names",
                     color="avg_temp", color_continuous_scale="RdYlBu_r",
                     title="Average Temperature by Country")
fig.show()


## 8. Summary, Insights & Deliverable Checklist

### Save artifacts for the report

In [ ]:
results_df.to_csv("model_comparison_results.csv", index=False)
country_avg.to_csv("country_climate_summary.csv", index=False)
print("Saved: model_comparison_results.csv, country_climate_summary.csv")
results_df


### Key Insights (fill in after running the notebook on your data)
- **Best-performing model:** _(pull the top row of `results_df` by RMSE)_
- **Strongest correlations:** _(read off the correlation heatmap in Section 3)_
- **Anomaly rate:** _(Isolation Forest flagged ~2% of rows by design — note any
  geographic or seasonal clustering you observe)_
- **Top predictive feature:** _(compare Sections 7.3's three importance methods —
  do they agree?)_
- **Air-quality takeaway:** _(which weather variable correlates most with PM2.5/PM10/etc.?)_
- **Regional takeaway:** _(which countries/continents run hottest, wettest, etc.?)_

### Limitations
- The dataset is a daily snapshot per location rather than a long multi-year
  series, which caps how much seasonality/trend a forecasting model can learn —
  call this out explicitly in your report rather than overstating model confidence.
- Air-quality columns are point-in-time and may be sparse for some locations.

### Submission Checklist (per the assessment doc)
- [ ] PM Accelerator mission statement displayed verbatim (Section 0 above)
- [ ] Report/presentation/dashboard covering cleaning, EDA, models, advanced
      analyses, and insights
- [ ] Public (or private + collaborators `community@pmaccelerator.io`,
      `hr@pmaccelerator.io`) GitHub repo, clone/download enabled
- [ ] `README.md` explaining project, methodology, and results
- [ ] `requirements.txt` listing all libraries used
- [ ] 1–2 min demo video (screen recording) walking through code + outputs,
      link shared in the submission
- [ ] Submitted via the Google form within the deadline
